<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# Notebook 02 – Tiền xử lý & Feature Engineering
**Đề 2: Dự đoán Bệnh Tim**

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from src.data.loader import DataLoader, load_config
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder

cfg = load_config('../configs/params.yaml')
loader = DataLoader('../configs/params.yaml')
print("Modules OK")

FileNotFoundError: [Errno 2] No such file or directory: '../configs/params.yaml'

## 1. Tải dữ liệu gốc

In [ ]:
df_raw = loader.load_raw()
print(f"Shape gốc: {df_raw.shape}")
print(df_raw.isnull().sum())

## 2. Tiền xử lý (DataCleaner)

In [ ]:
cleaner = DataCleaner(cfg)
df_processed = cleaner.fit_transform(df_raw)
print(f"Shape sau xử lý: {df_processed.shape}")
cleaner.print_before_after_stats(df_raw, df_processed)

## 3. Thống kê trước và sau xử lý

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (data, title) in zip(axes, [(df_raw, 'Trước xử lý'), (df_processed, 'Sau xử lý')]):
    data[['age','chol','trestbps','thalach','oldpeak']].boxplot(ax=ax)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../outputs/figures/preprocess_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering

In [ ]:
builder = FeatureBuilder(cfg)
# Rời rạc hoá cho Association Rules
df_disc = builder.discretize_all(df_raw)
print("Đặc trưng rời rạc mới:")
disc_cols = ['age_bin', 'chol_bin', 'trestbps_bin', 'thalach_bin', 'oldpeak_bin']
print(df_disc[disc_cols].head(10))

In [ ]:
# Đặc trưng tổng hợp nguy cơ
df_rich = builder.build_risk_features(df_processed)
print("Đặc trưng tổng hợp nguy cơ:")
risk_cols = ['age_sex_risk', 'chol_bp_ratio', 'hr_reserve', 'exang_oldpeak', 'cp_exang']
print(df_rich[risk_cols].describe().round(3))

## 5. Kiểm tra cân bằng lớp

In [ ]:
print("Phân phối target sau xử lý:")
print(df_processed['target'].value_counts())
print(f"Imbalance: {df_processed['target'].value_counts()[0]/df_processed['target'].value_counts()[1]:.2f}:1")
print("-> Khá cân bằng, SMOTE sẽ được dùng như bảo đảm thêm trong modeling.")

## 6. Lưu dữ liệu đã xử lý

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_processed.to_parquet('../data/processed/heart_processed.parquet', index=False)
print(f"Đã lưu: data/processed/heart_processed.parquet, shape={df_processed.shape}")

# Lưu thêm df với features tổng hợp
df_rich.to_parquet('../data/processed/heart_rich_features.parquet', index=False)
print(f"Đã lưu: data/processed/heart_rich_features.parquet")

In [ ]:
print("Notebook 02 hoàn tất.")